<a href="https://colab.research.google.com/github/RidhiSood22/neural-network-with-learnable-gates/blob/main/neural_network_with_learnable_gates.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import torchvision
import torchvision.transforms as transforms
from torch.utils.data import DataLoader
import matplotlib.pyplot as plt

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)

Using device: cpu


In [ ]:
transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.5,0.5,0.5),(0.5,0.5,0.5))
])

train_dataset = torchvision.datasets.CIFAR10(
    root='./data',
    train=True,
    download=True,
    transform=transform
)

test_dataset = torchvision.datasets.CIFAR10(
    root='./data',
    train=False,
    download=True,
    transform=transform
)

train_loader = DataLoader(train_dataset, batch_size=64, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=64, shuffle=False)

print("Train size:", len(train_dataset))
print("Test size:", len(test_dataset))

Train size: 50000
Test size: 10000


In [ ]:
class PrunableLinear(nn.Module):
    def __init__(self, in_features, out_features):
        super().__init__()

        self.weight = nn.Parameter(torch.randn(out_features, in_features)*0.01)
        self.bias = nn.Parameter(torch.zeros(out_features))

        self.gate_scores = nn.Parameter(torch.randn(out_features, in_features)-3)

    def forward(self, x):
        gates = torch.sigmoid(self.gate_scores*5)
        pruned_weights = self.weight * gates
        return F.linear(x, pruned_weights, self.bias)

In [ ]:
class PrunableNet(nn.Module):
    def __init__(self):
        super().__init__()
        self.fc1 = PrunableLinear(32*32*3, 512)
        self.fc2 = PrunableLinear(512, 256)
        self.fc3 = PrunableLinear(256, 10)

    def forward(self, x):
        x = x.view(x.size(0), -1)
        x = F.relu(self.fc1(x))
        x = F.relu(self.fc2(x))
        x = self.fc3(x)
        return x

In [ ]:
def sparsity_loss(model):
    loss = 0
    for module in model.modules():
        if isinstance(module, PrunableLinear):
            gates = torch.sigmoid(module.gate_scores)
            loss += gates.sum()   # better scaling
    return loss

In [ ]:
def train_model(lambda_val, epochs=5):
    model = PrunableNet().to(device)
    optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)

    for epoch in range(epochs):
        model.train()
        total_loss = 0

        for images, labels in train_loader:
            images, labels = images.to(device), labels.to(device)

            outputs = model(images)
            ce_loss = F.cross_entropy(outputs, labels)
            sp_loss = sparsity_loss(model)

            loss = ce_loss + lambda_val * sp_loss

            optimizer.zero_grad()
            loss.backward()
            optimizer.step()

            total_loss += loss.item()

        print(f"Epoch {epoch+1}, Loss: {total_loss:.2f}")

    return model

In [ ]:
def evaluate(model):
    model.eval()
    correct, total = 0, 0

    with torch.no_grad():
        for images, labels in test_loader:
            images, labels = images.to(device), labels.to(device)
            outputs = model(images)
            _, predicted = torch.max(outputs, 1)

            total += labels.size(0)
            correct += (predicted == labels).sum().item()

    return 100 * correct / total

In [ ]:
def calculate_sparsity(model, threshold=1e-2):
    total, zero = 0, 0

    for module in model.modules():
        if isinstance(module, PrunableLinear):
            gates = torch.sigmoid(module.gate_scores*2)

            total += gates.numel()
            zero += (gates < threshold).sum().item()

    return 100 * zero / total

In [ ]:
lambdas = [1e-2, 5e-2, 1e-1, 5e-1]
results = []

best_model = None
best_acc = 0

for lam in lambdas:
    print(f"\nTraining with lambda = {lam}")

    model = train_model(lam)
    acc = evaluate(model)
    sparsity = calculate_sparsity(model)

    results.append((lam, acc, sparsity))

    if acc > best_acc:
        best_acc = acc
        best_model = model

    print(f"Accuracy: {acc:.2f}%, Sparsity: {sparsity:.2f}%")


Training with lambda = 0.01
Epoch 1, Loss: 686319.65
Epoch 2, Loss: 389853.21
Epoch 3, Loss: 241084.81
Epoch 4, Loss: 157738.14
Epoch 5, Loss: 107130.45
Accuracy: 10.00%, Sparsity: 100.00%

Training with lambda = 0.05
Epoch 1, Loss: 3426602.21
Epoch 2, Loss: 1943231.29
Epoch 3, Loss: 1198930.94
Epoch 4, Loss: 781952.52
Epoch 5, Loss: 528763.72
Accuracy: 10.00%, Sparsity: 100.00%

Training with lambda = 0.1
Epoch 1, Loss: 6846536.16
Epoch 2, Loss: 3882128.31
Epoch 3, Loss: 2394500.66
Epoch 4, Loss: 1561057.21
Epoch 5, Loss: 1054999.15
Accuracy: 10.00%, Sparsity: 100.00%

Training with lambda = 0.5


In [ ]:
all_gates = []

for module in best_model.modules():
    if isinstance(module, PrunableLinear):
        gates = torch.sigmoid(module.gate_scores).detach().cpu().numpy()
        all_gates.extend(gates.flatten())

plt.hist(all_gates, bins=50)
plt.title("Gate Distribution")
plt.xlabel("Gate Value")
plt.ylabel("Frequency")
plt.show()

In [ ]:
print("\nFinal Results:")
for r in results:
    print(f"Lambda={r[0]}, Accuracy={r[1]:.2f}%, Sparsity={r[2]:.2f}%")